# 04 — Export del modelo final

Entrena el modelo sobre el dataset completo (sin split) y guarda el bundle en `src/models/rf_v1.pkl`.

> Ejecutar **solo** si `03-training.ipynb` pasó todos los criterios de aceptación.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

PROCESSED_PATH = Path("../data/processed/train_data.csv")
OUTPUT_PATH = Path("../../../src/models/rf_v1.pkl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {0: "adequate", 1: "forward_slouch", 2: "excessive_recline"}

## 1. Carga del dataset completo

In [ ]:
df = pd.read_csv(PROCESSED_PATH)
X = df[["Ax1", "Ay1", "Az1"]].values
y = df["label"].values
print(f"Dataset completo: {X.shape[0]} muestras, {X.shape[1]} features")

## 2. Entrenamiento del modelo final

In [ ]:
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X, y)
print("Modelo final entrenado.")

## 3. Guardado del bundle

In [ ]:
bundle = {
    "model": clf,
    "label_map": CLASS_NAMES,
    "features": ["Ax1", "Ay1", "Az1"],
    "model_version": "rf_v1",
    "feature_description": "dorsal sensor accelerometer (ax, ay, az)",
}
joblib.dump(bundle, OUTPUT_PATH)
size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"Guardado en: {OUTPUT_PATH}")
print(f"Tamaño: {size_mb:.2f} MB")

## 4. Verificación del bundle

In [ ]:
loaded = joblib.load(OUTPUT_PATH)
print("Claves del bundle:", list(loaded.keys()))
print("Label map:", loaded["label_map"])
print("Features:", loaded["features"])

## 5. Smoke test del servicio

In [ ]:
test_cases = [
    ([0.05, 9.8, 0.1], "adequate — sentado erguido"),
    ([0.3, 7.5, 2.1], "forward_slouch — encorvado hacia adelante"),
    ([-0.2, 6.0, -3.5], "excessive_recline — reclinado hacia atrás"),
]

model = loaded["model"]
label_map = loaded["label_map"]

print("Smoke test:")
print("-" * 55)
for features, description in test_cases:
    proba = model.predict_proba([features])[0]
    idx = int(proba.argmax())
    cls = label_map[idx]
    conf = proba[idx]
    print(f"Input: {features}")
    print(f"  → {cls} (confianza: {conf:.3f}) | {description}")
    print()

## Listo

El modelo `rf_v1.pkl` está en `src/models/`. Para levantar el servicio:

```bash
# desde la raíz de sitright-ml-service
python -m venv venv
venv\Scripts\activate      # Windows
pip install -r requirements.txt
set MODEL_PATH=./src/models/rf_v1.pkl
uvicorn src.main:app --reload --port 8001
```